# QSAR Modeling — LC50 Prediction

Build regression models to predict log_LC50 from molecular descriptors and fingerprints with feature selection and train/test validation.

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import cross_val_predict, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load curated data

In [2]:
df = pd.read_csv('data/curated_data_rdkit.csv')
print(f'Dataset: {df.shape[0]} molecules, {df.shape[1]} columns')
print(f'Target: log_LC50 range [{df["log_LC50"].min():.2f}, {df["log_LC50"].max():.2f}]')
df.head()

Dataset: 2303 molecules, 126 columns
Target: log_LC50 range [-5.00, 4.76]


,CAS_str,Chemical_name,log_LC50,n_measurements,SMILES,mlecule_H_final,SlogP,SMR,LabuteASA,TPSA,...,MQN34,MQN35,MQN36,MQN37,MQN38,MQN39,MQN40,MQN41,MQN42,SMILES_curated
0,50-06-6,Phenobarbital,2.685294,2,CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,[H]c1c([H])c([H])c(C2(C([H])([H])C([H])([H])[H...,0.7004,60.0924,115.175275,75.27,...,0,0,2,0,0,0,0,0,0,[H]c1c([H])c([H])c(C2(C([H])([H])C([H])([H])[H...
1,50-18-0,Cyclophosphamide,3.340444,1,C1CNP(=O)(OC1)N(CCCl)CCCl,[H]N1C([H])([H])C([H])([H])C([H])([H])OP1(=O)N...,1.8840,59.1912,115.574149,41.57,...,0,0,1,0,0,0,0,0,0,[H]N1C([H])([H])C([H])([H])C([H])([H])OP1(=O)N...
2,50-28-2,17beta-Estradiol,0.330673,5,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=C3...,[H]Oc1c([H])c([H])c2c(c1[H])C([H])([H])C([H])(...,3.6092,78.7306,154.274315,40.46,...,0,1,3,0,0,0,0,6,3,[H]O[C@@]1([H])C([H])([H])C([H])([H])[C@@]2([H...
3,50-29-3,"p,p'-DDT",-1.737289,448,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)(Cl)Cl)Cl,[H]c1c([H])c(C([H])(c2c([H])c([H])c(Cl)c([H])c...,6.4955,85.0370,149.379447,0.00,...,0,0,2,0,0,0,0,0,0,[H]c1c([H])c(C([H])(c2c([H])c([H])c(Cl)c([H])c...
4,50-30-6,"2,6-Dichlorobenzoic acid",2.112655,6,C1=CC(=C(C(=C1)Cl)C(=O)O)Cl,[H]OC(=O)c1c(Cl)c([H])c([H])c([H])c1Cl,2.6916,43.4213,79.065186,37.30,...,0,0,1,0,0,0,0,0,0,[H]O=C(O)c1c(Cl)c([H])c([H])c([H])c1Cl


## 2. Generate Morgan (ECFP) fingerprints

In [3]:
mols = df['SMILES'].apply(Chem.MolFromSmiles)
valid = mols.notna()
mols = mols[valid]
print(f'Valid molecules: {len(mols)} / {len(df)}')

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
fp_array = np.array(fps)
print(f'Fingerprint matrix: {fp_array.shape} (radius=2, 2048 bits)')

Valid molecules: 2303 / 2303
Fingerprint matrix: (2303, 2048) (radius=2, 2048 bits)


## 3. Prepare features

In [4]:
meta_cols = ['CAS_str', 'Chemical_name', 'SMILES', 'mlecule_H_final', 'SMILES_curated']
target_col = 'log_LC50'
desc_cols = [c for c in df.columns if c not in meta_cols + [target_col, 'n_measurements']]

df_model = df[valid].reset_index(drop=True)
X_desc = df_model[desc_cols].values
X_fp = fp_array
X_combined = np.hstack([X_desc, X_fp])
y = df_model[target_col].values
all_feature_names = desc_cols + [f'FP_{i}' for i in range(X_fp.shape[1])]

print(f'Descriptors: {X_desc.shape[1]}')
print(f'Fingerprints: {X_fp.shape[1]}')
print(f'Combined features: {X_combined.shape[1]}')
print(f'Samples: {X_combined.shape[0]}')

Descriptors: 119
Fingerprints: 2048
Combined features: 2167
Samples: 2303


## 4. Feature filtering — Variance threshold

In [5]:
var_thresh = VarianceThreshold(threshold=0.01)
X_var_filtered = var_thresh.fit_transform(X_combined)
kept_mask = var_thresh.get_support()
kept_names = [all_feature_names[i] for i in range(len(all_feature_names)) if kept_mask[i]]

removed = len(all_feature_names) - len(kept_names)
print(f'Features before: {len(all_feature_names)}')
print(f'Features removed (variance < 0.01): {removed}')
print(f'Features remaining: {X_var_filtered.shape[1]}')

Features before: 2167
Features removed (variance < 0.01): 1633
Features remaining: 534


## 5. Feature filtering — Remove highly correlated features

In [6]:
corr_threshold = 0.95

corr_matrix = pd.DataFrame(X_var_filtered).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > corr_threshold)]

keep_idx = [i for i in range(X_var_filtered.shape[1]) if i not in to_drop]
X_final = X_var_filtered[:, keep_idx]
final_names = [kept_names[i] for i in keep_idx]

print(f'Features after variance filter: {X_var_filtered.shape[1]}')
print(f'Correlated features removed (|r| > {corr_threshold}): {len(to_drop)}')
print(f'Final feature count: {X_final.shape[1]}')

Features after variance filter: 534
Correlated features removed (|r| > 0.95): 21
Final feature count: 513


## 6. Define models

In [7]:
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    'SVR': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'PLS': PLSRegression(n_components=20),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'Bagging': BaggingRegressor(n_estimators=100, max_samples=0.8, n_jobs=-1, random_state=42),
}

## 7. Cross-validation evaluation (5-fold)

In [8]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
cv_predictions = {}

for name, model in models.items():
    X = X_final.copy()
    if name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    y_pred = cross_val_predict(model, X, y, cv=cv)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    
    results.append({'Model': name, 'R²': r2, 'RMSE': rmse, 'MAE': mae})
    cv_predictions[name] = y_pred
    print(f'{name:20s}  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')

results_df = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
print()
print(results_df.to_string(index=False))

Ridge                 R²=0.2392  RMSE=1.1597  MAE=0.8460


Lasso                 R²=0.3999  RMSE=1.0300  MAE=0.7610


ElasticNet            R²=0.3713  RMSE=1.0542  MAE=0.7741


SVR                   R²=0.4719  RMSE=0.9662  MAE=0.7054
KNN                   R²=0.2932  RMSE=1.1178  MAE=0.8397


PLS                   R²=0.2466  RMSE=1.1541  MAE=0.8400


...

GradientBoosting      R²=0.5192  RMSE=0.9219  MAE=0.6549


XGBoost               R²=0.5246  RMSE=0.9167  MAE=0.6481


LightGBM              R²=0.5318  RMSE=0.9097  MAE=0.6548


AdaBoost              R²=0.3489  RMSE=1.0728  MAE=0.8177


Bagging               R²=0.5184  RMSE=0.9227  MAE=0.6588

           Model       R²     RMSE      MAE
        LightGBM 0.531846 0.909733 0.654759
         XGBoost 0.524626 0.916722 0.648061
    RandomForest 0.521533 0.919700 0.655443
      ExtraTrees 0.520398 0.920789 0.642282
GradientBoosting 0.519224 0.921916 0.654870
         Bagging 0.518372 0.922733 0.658807
             SVR 0.471913 0.966212 0.705420
           Lasso 0.399874 1.030010 0.761016
      ElasticNet 0.371344 1.054208 0.774068
        AdaBoost 0.348935 1.072833 0.817682
             KNN 0.293225 1.117791 0.839667
             PLS 0.246609 1.154065 0.840031
           Ridge 0.239168 1.159750 0.846004


## 8. Feature set comparison with best model

In [9]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

feature_sets = {
    'Raw combined': X_combined,
    'Variance filtered': X_var_filtered,
    'Variance + Correlation filtered': X_final,
}

print(f'Model: {best_model_name}\n')
comparison = []

for fname, X_feat in feature_sets.items():
    X = X_feat.copy()
    if best_model_name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    y_pred = cross_val_predict(best_model, X, y, cv=cv)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    
    comparison.append({'Features': fname, 'R²': r2, 'RMSE': rmse, 'MAE': mae})
    print(f'{fname:35s}  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')

print()
print(pd.DataFrame(comparison).to_string(index=False))

Model: LightGBM



Raw combined                         R²=0.5298  RMSE=0.9117  MAE=0.6508


Variance filtered                    R²=0.5327  RMSE=0.9089  MAE=0.6483


Variance + Correlation filtered      R²=0.5318  RMSE=0.9097  MAE=0.6548

                       Features       R²     RMSE      MAE
                   Raw combined 0.529799 0.911721 0.650816
              Variance filtered 0.532669 0.908934 0.648344
Variance + Correlation filtered 0.531846 0.909733 0.654759


## 9. Train/test split — final validation

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

Train set: 1842 samples
Test set:  461 samples


In [11]:
os.makedirs('models', exist_ok=True)

test_results = []

for name, model in models.items():
    X_tr = X_train.copy()
    X_te = X_test.copy()
    
    if name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        joblib.dump(scaler, f'models/{name}_scaler.pkl')
    
    model.fit(X_tr, y_train)
    
    y_train_pred = model.predict(X_tr)
    y_test_pred = model.predict(X_te)
    
    train_r2 = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    test_results.append({
        'Model': name, 'Train R²': train_r2, 'Train RMSE': train_rmse,
        'Test R²': test_r2, 'Test RMSE': test_rmse, 'Test MAE': test_mae,
        'Gap': train_r2 - test_r2
    })
    
    joblib.dump(model, f'models/{name}_trained.pkl')
    print(f'{name:20s}  Train R²={train_r2:.4f}  Test R²={test_r2:.4f}  Gap={train_r2 - test_r2:.4f}')

test_results_df = pd.DataFrame(test_results).sort_values('Test R²', ascending=False).reset_index(drop=True)
print()
print(test_results_df.to_string(index=False))

Ridge                 Train R²=0.6499  Test R²=0.2075  Gap=0.4424
Lasso                 Train R²=0.5805  Test R²=0.3652  Gap=0.2153


ElasticNet            Train R²=0.6116  Test R²=0.3418  Gap=0.2698


SVR                   Train R²=0.7546  Test R²=0.4443  Gap=0.3103
KNN                   Train R²=0.5384  Test R²=0.2459  Gap=0.2925
PLS                   Train R²=0.6362  Test R²=0.2215  Gap=0.4148


RandomForest          Train R²=0.9346  Test R²=0.4730  Gap=0.4616


ExtraTrees            Train R²=0.9961  Test R²=0.4685  Gap=0.5276


GradientBoosting      Train R²=0.9146  Test R²=0.4819  Gap=0.4327


XGBoost               Train R²=0.9426  Test R²=0.5093  Gap=0.4333


LightGBM              Train R²=0.8678  Test R²=0.5078  Gap=0.3600


AdaBoost              Train R²=0.4261  Test R²=0.3035  Gap=0.1226


Bagging               Train R²=0.9021  Test R²=0.4685  Gap=0.4336

           Model  Train R²  Train RMSE  Test R²  Test RMSE  Test MAE      Gap
         XGBoost  0.942598    0.319537 0.509274   0.918020  0.642206 0.433324
        LightGBM  0.867842    0.484847 0.507802   0.919396  0.650740 0.360040
GradientBoosting  0.914624    0.389696 0.481918   0.943260  0.653537 0.432706
    RandomForest  0.934596    0.341083 0.472964   0.951377  0.668893 0.461632
         Bagging  0.902107    0.417285 0.468550   0.955352  0.675289 0.433557
      ExtraTrees  0.996128    0.082992 0.468522   0.955377  0.650951 0.527605
             SVR  0.754598    0.660688 0.444335   0.976875  0.707695 0.310263
           Lasso  0.580478    0.863843 0.365192   1.044128  0.764937 0.215287
      ElasticNet  0.611596    0.831188 0.341842   1.063158  0.776022 0.269755
        AdaBoost  0.426098    1.010361 0.303504   1.093683  0.830056 0.122594
             KNN  0.538358    0.906171 0.245861   1.138042  0.840211 0.2924

## 10. Best model — detailed test analysis

In [12]:
best_test_name = test_results_df.iloc[0]['Model']
best_test_model = models[best_test_name]

X_tr = X_train.copy()
X_te = X_test.copy()
if best_test_name in ['Ridge', 'Lasso', 'ElasticNet', 'SVR', 'KNN', 'PLS']:
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

best_test_model.fit(X_tr, y_train)
y_test_pred = best_test_model.predict(X_te)

print(f'Best model: {best_test_name}')
print(f'Test R²:  {r2_score(y_test, y_test_pred):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}')
print(f'Test MAE:  {mean_absolute_error(y_test, y_test_pred):.4f}')

Best model: XGBoost
Test R²:  0.5093
Test RMSE: 0.9180
Test MAE:  0.6422


### Predicted vs Actual (test set)

In [13]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_test_pred, alpha=0.5, edgecolors='none')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual log_LC50')
axes[0].set_ylabel('Predicted log_LC50')
axes[0].set_title(f'{best_test_name} - Predicted vs Actual')
axes[0].set_aspect('equal')

residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.5, edgecolors='none')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted log_LC50')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.savefig(f'figures/abdek_figures/{best_test_name}_parity_plot.png', dpi=150)
plt.show()

## 11. Feature importance (tree-based best model)

In [14]:
tree_models = ['RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM']

if best_test_name in tree_models:
    importances = best_test_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': final_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f'Top 20 features by importance ({best_test_name}):')
    print(feat_imp.head(20).to_string(index=False))
    
    feat_imp.to_csv(f'models/{best_test_name}_feature_importance.csv', index=False)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    top20 = feat_imp.head(20)
    ax.barh(range(len(top20)), top20['Importance'].values, align='center')
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20['Feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Features — {best_test_name}')
    plt.tight_layout()
    plt.savefig(f'figures/abdek_figures/{best_test_name}_feature_importance.png', dpi=150)
    plt.show()
else:
    print(f'{best_test_name} does not support feature importance.')

Top 20 features by importance (XGBoost):
    Feature  Importance
     FP_534    0.089704
     FP_827    0.052075
       MQN3    0.048458
    FP_1476    0.030467
    FP_1764    0.016913
    FP_1384    0.015173
      MQN11    0.014072
     FP_991    0.011337
    FP_1356    0.009855
    FP_1729    0.009772
       MQN7    0.009582
      FP_91    0.008502
slogp_VSA12    0.008276
     FP_753    0.008125
     FP_549    0.008063
    FP_2036    0.008007
       MQN6    0.008006
    FP_1223    0.007812
     FP_504    0.007561
      MQN14    0.007440


## 12. Save summary report

In [15]:
report = f"""QSAR Modeling Summary
{'='*50}
Dataset: {len(y)} molecules
Target: log_LC50
Features after filtering: {X_final.shape[1]} (from {X_combined.shape[1]})
  - Variance filter (threshold=0.01): removed {len(all_feature_names) - len(kept_names)} features
  - Correlation filter (|r|>{corr_threshold}): removed {len(to_drop)} features
Train/test split: 80/20 (random_state=42)

Best model: {best_test_name}
  Test R²:  {test_results_df.iloc[0]['Test R²']:.4f}
  Test RMSE: {test_results_df.iloc[0]['Test RMSE']:.4f}
  Test MAE:  {test_results_df.iloc[0]['Test MAE']:.4f}

All models saved to: models/
"""

with open('models/modeling_summary.txt', 'w') as f:
    f.write(report)

print(report)

QSAR Modeling Summary
Dataset: 2303 molecules
Target: log_LC50
Features after filtering: 513 (from 2167)
  - Variance filter (threshold=0.01): removed 1633 features
  - Correlation filter (|r|>0.95): removed 21 features
Train/test split: 80/20 (random_state=42)

Best model: XGBoost
  Test R²:  0.5093
  Test RMSE: 0.9180
  Test MAE:  0.6422

All models saved to: models/

